# Vehicle Detection, Tracking, and Trajectory Extraction

## G-TRISP Internship Task – Group B1

This project implements a traffic video analytics pipeline using a pretrained YOLOv8 model for:
- vehicle detection
- multi-object tracking
- trajectory extraction

The system processes a selected portion of the traffic video and generates:
- an annotated output video
- a structured trajectory CSV file

In [13]:
import cv2
import pandas as pd
import numpy as np
import time

from ultralytics import YOLO, RTDETR

import warnings
warnings.filterwarnings("ignore")

## Loading Pretrained YOLOv8 Model and Traffic Video

The lightweight `yolov8n` pretrained model is used for vehicle detection and tracking.

The traffic video is loaded using OpenCV for frame-by-frame processing.

In [ ]:
model = YOLO("yolov8n.pt") 
# model = YOLO("yolov8x.pt")
# model = YOLO("yolo11n.pt") # Latest stable (YOLOv11 Nano)
# model = RTDETR("rtdetr-l.pt") # Real-Time Detection Transformer

VIDEO_PATH = "../data/HighWay.mp4"

cap = cv2.VideoCapture(VIDEO_PATH)

## Extracting Video Metadata

Video properties such as:
- frame dimensions
- FPS
- total frame count

are extracted for processing and output video generation.

In [ ]:
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fps = int(cap.get(cv2.CAP_PROP_FPS))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print("Width:", frame_width)
print("Height:", frame_height)
print("FPS:", fps)
print("Total Frames:", total_frames)

Width: 1920
Height: 1080
FPS: 30
Total Frames: 1022


## Defining Processing Range and Output Video

The system allows processing only a selected segment of the video using:
- START_FRAME
- END_FRAME

An output video writer is initialized to save annotated tracking results.

In [16]:
START_FRAME = 0
END_FRAME = 1022

fourcc = cv2.VideoWriter_fourcc(*'XVID')

out = cv2.VideoWriter(
    "../outputs/annotated_video.avi",
    fourcc,
    fps,
    (frame_width, frame_height)
)

## Initializing Trajectory Storage

A list is created to store frame-wise trajectory information including:
- frame number
- tracking ID
- vehicle class
- bounding box coordinates
- centroid coordinates

In [17]:
trajectory_data = []

frame_count = 0

## Vehicle Detection and Tracking

The traffic video is processed frame-by-frame.

For each frame:
- YOLOv8 performs vehicle detection
- tracking IDs are assigned
- bounding boxes are drawn
- trajectory information is stored

In [18]:
print(f"Starting processing with model: {model.ckpt_path}")
start_time = time.perf_counter()
processed_frames = 0

while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame_count += 1
    processed_frames += 1

    if frame_count < START_FRAME:
        continue

    if frame_count > END_FRAME:
        break

    results = model.track(
        frame,
        persist=True,
        verbose=False
    )

    boxes = results[0].boxes

    for box in boxes:

        x1, y1, x2, y2 = box.xyxy[0].tolist()
        x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])

        if box.id is not None:
            track_id = int(box.id[0])
        else:
            track_id = -1

        class_id = int(box.cls[0])
        class_name = model.names[class_id]

        centroid_x = int((x1 + x2) / 2)
        centroid_y = int((y1 + y2) / 2)

        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            2
        )

        label = f"{class_name} ID:{track_id}"

        cv2.putText(
            frame,
            label,
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 255, 0),
            2
        )

        trajectory_data.append({
            "frame": frame_count,
            "track_id": track_id,
            "vehicle_class": class_name,
            "x1": x1,
            "y1": y1,
            "x2": x2,
            "y2": y2,
            "centroid_x": centroid_x,
            "centroid_y": centroid_y
        })

    out.write(frame)

end_time = time.perf_counter()
total_time = end_time - start_time
print(f"Processing Finished.")
print(f"Total Time: {total_time:.2f} seconds")
print(f"Average FPS: {processed_frames / total_time:.2f}")

Starting processing with model: rtdetr-l.pt
Processing Finished.
Total Time: 125.02 seconds
Average FPS: 8.17


## Releasing Video Resources

Video resources are released after processing completion.

In [19]:
cap.release()
out.release()

cv2.destroyAllWindows()

## Exporting Trajectory Data

The collected trajectory information is converted into a structured Pandas DataFrame and exported as CSV.

In [20]:
trajectory_df = pd.DataFrame(trajectory_data)

trajectory_df.to_csv(
    "../outputs/trajectories.csv",
    index=False
)

trajectory_df.head()

,frame,track_id,vehicle_class,x1,y1,x2,y2,centroid_x,centroid_y
0,1,1,bus,653,448,851,728,752,588
1,1,2,car,574,738,737,940,655,839
2,1,3,car,182,846,429,1079,305,962
3,1,4,car,1394,622,1510,741,1452,681
4,1,5,car,292,555,441,675,366,615
